# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os
if not os.path.exists("flyrank-internship"):
    !git clone https://github.com/maheen-armghan/flyrank-internship.git
os.chdir("flyrank-internship")
print("Now in:", os.getcwd())

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 348, done.
remote: Counting objects: 100% (348/348), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 348 (delta 192), reused 293 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (348/348), 1.94 MiB | 9.13 MiB/s, done.
Resolving deltas: 100% (192/192), done.
Now in: /content/flyrank-internship


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print("Rows:", len(df))

Rows: 30000


**Method: Random Forest classifier**, compared against Logistic Regression and a single Decision Tree as intermediate steps.

Why: my lane (Refresh / Content Opportunity Scoring) is a classification problem (declining vs. not) whose output feeds a ranking (Precision@K), which is exactly the setup Week 1's starter notebooks demonstrated — a random forest clearly beat both a hand rule and a single tree on Precision@50 there. My Week 4 baseline was a simple rule (impressions x decline flag); a random forest can combine multiple signals (position, CTR, impressions, staleness) the way the baseline can't, which is the direction Section 5 of w02_ml_task_framing already argued for. I'm not reaching for gradient boosting yet since the dataset is a small (~30k row) anonymized sample — that extra complexity isn't earned unless a simpler model clearly underfits.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ["search_volume", "competition", "cpc", "word_count", "content_age_days",
            "impressions_90d", "sessions_90d", "avg_position", "ctr",
            "days_since_last_update", "engagement_rate", "scroll_rate"]
features = [f for f in features if f in df.columns]
print("Using features:", features)

Using features: ['search_volume', 'competition', 'cpc', 'word_count', 'content_age_days', 'impressions_90d', 'sessions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'engagement_rate', 'scroll_rate']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: client-holdout** (grouped split by `client_id`), not a plain random split, this matches the guide's validation rules (Section 12): pages from the same client can share patterns the model could memorize, so testing on unseen clients gives an honest measure of generalization. This is the same split strategy the reference pipeline (`scripts/03_train_model.py`) uses.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"] if "client_id" in df.columns else df["content_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(X_train), " Test rows:", len(X_test))
print("Unique clients in train/test overlap check:",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])), "(should be 0)")

Train rows: 19166  Test rows: 10834
Unique clients in train/test overlap check: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Comparing my Week 4 baseline score against Logistic Regression, a single Decision Tree, and a Random Forest — all evaluated on the same held-out test split using Precision@50, matching the metric chosen in w02's framing.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Baseline score, recomputed on the test split only
test_df = df.iloc[test_idx].copy()
baseline_score_test = ((test_df["is_declining_label"] == 1) & (test_df["impressions_90d"] >= 100)).astype(int) * test_df["impressions_90d"]

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42),
}

results = {"Baseline (Week 4 rule)": precision_at_k(baseline_score_test.values, y_test.values, 50)}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = precision_at_k(proba, y_test.values, 50)

results_table = pd.DataFrame(results.items(), columns=["Method", "Precision@50"]).sort_values("Precision@50", ascending=False)
print(results_table.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                Method  Precision@50
Baseline (Week 4 rule)          1.00
   Logistic Regression          0.60
         Decision Tree          0.60
         Random Forest          0.58


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
best_model = models["Random Forest"]
importances = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

impressions_90d           0.314348
avg_position              0.207002
content_age_days          0.179019
word_count                0.107159
days_since_last_update    0.047299
scroll_rate               0.046957
ctr                       0.039068
sessions_90d              0.030223
search_volume             0.012018
engagement_rate           0.007678
competition               0.005234
cpc                       0.003996
dtype: float64


In [7]:
proba_rf = best_model.predict_proba(X_test)[:, 1]
test_df["rf_score"] = proba_rf
top50 = test_df.sort_values("rf_score", ascending=False).head(50)
misses = top50[top50["is_declining_label"] == 0]
print(f"{len(misses)} of the top 50 were NOT actually declining (false positives).")
misses[["content_id","impressions_90d","avg_position","ctr","trend_direction","rf_score"]].head(10)

21 of the top 50 were NOT actually declining (false positives).


,content_id,impressions_90d,avg_position,ctr,trend_direction,rf_score
27035,content_9c128be31943,1296,23.8,0.15,up,0.784842
20736,content_41baf0722ad9,3115,12.8,0.00,stable,0.784594
15817,content_f79387f83703,1945,12.4,0.15,stable,0.783090
11061,content_0b47dae0c7f9,1191,23.1,0.00,stable,0.782590
22524,content_846bb4dd8b44,870,17.6,0.11,stable,0.780023
12332,content_4d9f36001f06,3369,13.2,0.03,stable,0.775448
19475,content_ad73299cdf36,1144,10.5,0.61,stable,0.774885
28718,content_ef6e7d7cfe15,264,22.2,0.00,stable,0.774843
11376,content_5ebe639dafce,8979,19.6,0.17,stable,0.773569
10080,content_35d63627bf3e,1525,32.6,0.00,stable,0.772247


**Model vs baseline:** [fill with your real results_table — did RF beat the baseline? by how much?]

**Top features:** [name the top 2-3 from your importances output] carried the most weight, consistent with Signal 2's finding that CTR/position interact strongly with decline.

**Errors:** [N] of the top 50 predictions were false positives. Looking at these misses, [describe any pattern you see — e.g. "several have strong CTR and good position despite being labeled declining, suggesting the current-window proxy label itself may be noisy for these edge cases, echoing the caveat flagged back in w02_ml_task_framing."]

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.